# USearch Jaccard: Binary kNN without MinHash

TMAP now uses USearch's native Jaccard distance on bit-packed vectors for binary
fingerprint data. This replaces the MinHash+LSH pipeline for dense binary matrices,
giving higher recall (98% vs 1%), lower memory (16x), and faster batch queries.

This notebook shows:
1. How the estimator auto-selects the USearch backend
2. Using USearchIndex standalone for kNN queries (no layout needed)
3. Incremental queries with `transform()` and `add_points()`

## 1. Estimator: Automatic Backend Selection

Pass a binary matrix to `TMAP(metric='jaccard')` and USearch is used automatically.

In [ ]:
import numpy as np
from tmap import TMAP

# Simulate 5000 ECFP-like fingerprints: 2048 bits, ~10% density
rng = np.random.default_rng(42)
fingerprints = (rng.random((5000, 2048)) < 0.1).astype(np.uint8)

model = TMAP(metric="jaccard", n_neighbors=20, seed=42).fit(fingerprints)
print(f"Embedding shape: {model.embedding_.shape}")
print(f"Backend: USearch (index_ available: {model._index is not None})")
print(f"LSH forest: {model._lsh_forest is not None}")

Sets and strings still use MinHash+LSH automatically:

In [ ]:
# Variable-length sets -> MinHash + LSH (automatic)
sets = [sorted(rng.choice(500, 30, replace=False).tolist()) for _ in range(200)]
model_sets = TMAP(metric="jaccard", n_neighbors=5, n_permutations=128, seed=42).fit(sets)
print(f"Backend: LSH (lsh_forest_ available: {model_sets._lsh_forest is not None})")

## 2. Standalone USearchIndex: kNN Without Layout

Use `USearchIndex` directly when you just need fast kNN queries — no OGDF or tree layout required.

In [ ]:
from tmap.index import USearchIndex

# Build index from binary fingerprints
idx = USearchIndex(seed=42, expansion_search=512)
idx.build_from_binary(fingerprints)
print(f"Index: {idx.n_nodes} points, metric={idx.metric}")

In [ ]:
# Query single point
neighbors, distances = idx.query_point(fingerprints[0], k=10)
print(f"10 nearest neighbors of point 0: {neighbors}")
print(f"Jaccard distances: {np.round(distances, 4)}")

In [ ]:
# Batch query — fast, vectorized
query = fingerprints[:100]
neighbors, distances = idx.query_batch(query, k=20)
print(f"Batch result shape: {neighbors.shape}")
print(f"Distance range: [{distances.min():.4f}, {distances.max():.4f}]")

In [ ]:
# Full kNN graph (all-vs-all)
knn = idx.query_knn(k=20)
print(f"kNN graph: {knn.indices.shape[0]} points x {knn.indices.shape[1]} neighbors")
print(f"All neighbors found: {(knn.indices >= 0).all()}")

In [ ]:
# Incremental: add new points to existing index
new_fps = (rng.random((50, 2048)) < 0.1).astype(np.uint8)
keys = idx.add(new_fps)
print(f"Added {len(keys)} points, index now has {idx.n_nodes} points")

# New points are immediately queryable
nb, dist = idx.query_point(new_fps[0], k=5)
print(f"Nearest to new point: {nb}")

In [ ]:
# Save and load
idx.save("/tmp/fingerprint_index.usearch")
restored = USearchIndex.load("/tmp/fingerprint_index.usearch")
print(f"Restored: {restored.n_nodes} points, metric={restored.metric}")

# Restored index supports all operations
nb2, dist2 = restored.query_point(new_fps[0], k=5)
print(f"Same results after load: {np.array_equal(nb, nb2)}")

## 3. Transform and Add Points

The USearch index is stored automatically for binary Jaccard, so `transform()` and
`add_points()` work without `store_index=True`.

In [ ]:
# transform: project new points onto existing map (non-mutating)
new_fps = (rng.random((20, 2048)) < 0.1).astype(np.uint8)
new_coords = model.transform(new_fps)
print(f"Projected {new_coords.shape[0]} new points: shape={new_coords.shape}")

In [ ]:
# add_points: insert new points into the model (mutating)
n_before = model.embedding_.shape[0]
added_coords = model.add_points(new_fps)
n_after = model.embedding_.shape[0]
print(f"Points: {n_before} -> {n_after} (+{n_after - n_before})")

## When to Use What

| Use case | Tool |
|---|---|
| Build a TMAP from fingerprints | `TMAP(metric='jaccard').fit(X)` — auto-selects USearch |
| Just need kNN, no layout | `USearchIndex().build_from_binary(X)` |
| Similarity search against a database | `idx.query_batch(queries, k=50)` |
| Incremental insertion | `idx.add(new_data)` |
| Sparse binary data (single-cell) | `TMAP(metric='jaccard').fit(csr_matrix)` — auto-selects LSH |
| Variable-length sets/strings | `TMAP(metric='jaccard').fit(list_of_sets)` — auto-selects LSH |